# 📡 Stream Sandbox Compute with VideoDB

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/hackathon/guides/sandbox/stream_sandbox_compute.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Use a **new VideoDB Sandbox** as dedicated compute for **RTStream visual indexing**, then receive real-time alert notifications in this notebook over **WebSocket**.

This notebook intentionally avoids webhook/callback setup. The whole test runs in the notebook:

1. 🏗️ Create a new sandbox.
2. 🔌 Open a WebSocket connection.
3. 📹 Connect a new RTSP sample stream.
4. 🧠 Create an RTStream visual index with only `model_name` + `sandbox_id`.
5. 🚨 Create alert rules with `ws_connection_id`.
6. 📩 Watch incoming `scene_index` and `alert` events.
7. 🧹 Clean up the alert, index, stream, WebSocket, and sandbox.


## 🛠️ 1. Install dependencies

Install the SDK and the optional WebSocket dependency.

In [ ]:
!pip install -q "git+https://github.com/Video-DB/videodb-python.git@hackathon"
!pip install -q "websockets" python-dotenv

## 🔑 2. Connect to VideoDB

In [ ]:
import os
import json
import asyncio
from datetime import datetime, timezone
from getpass import getpass

from videodb import connect, SandboxModel, SandboxTier

if not os.environ.get("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect()
coll = conn.get_collection()
print("Connected to VideoDB")

## 🤖 3. Choose a sandbox model

For lowest latency, start with the smallest visual model:

| Recommended order | Model | Suggested tier | Notes |
|---:|---|---|---|
| 1 | `SandboxModel.GEMMA_4_E2B` | `SandboxTier.small` | Best first choice for low-latency RTStream tests. |
| 2 | `SandboxModel.QWEN_9B` | `SandboxTier.small` | Larger small-tier VLM; try if you want different behavior. |
| 3 | `SandboxModel.GEMMA_4_26B` | `SandboxTier.medium` | Higher-capacity model, more latency. |
| 4 | `SandboxModel.QWEN_27B` | `SandboxTier.medium` | Higher-capacity model, more latency. |
| 5 | `SandboxModel.GEMMA_4_31B` | `SandboxTier.medium` | Largest listed VLM; expect the most latency. |

The notebook defaults to `GEMMA_4_E2B` because it should be the fastest of the listed visual models. To test another model, change `SELECTED_MODEL` and `SANDBOX_TIER` below before running the sandbox cell.

In [ ]:
# Pick the model you want to test.
# GEMMA_4_E2B is the default recommendation for lowest-latency RTStream testing.
SELECTED_MODEL = SandboxModel.GEMMA_4_E2B
SANDBOX_TIER = SandboxTier.small

print("Selected model:", SELECTED_MODEL)
print("Selected sandbox tier:", SANDBOX_TIER)


## 🏗️ 4. Create a new sandbox

This notebook always creates a fresh sandbox so each user gets an isolated test environment. The cleanup cell at the end stops it.

In [ ]:
sandbox = conn.create_sandbox(tier=SANDBOX_TIER)
print(f"Created sandbox: {sandbox.id} | status={sandbox.status} | tier={sandbox.tier}")

sandbox.wait_for_ready(timeout=300, interval=5)
print(f"Sandbox ready: {sandbox.id} | status={sandbox.status}")

## 🔌 5. Open a WebSocket connection

The WebSocket returns a `connection_id`. We pass this ID to the RTStream index and alerts so events arrive in this notebook.

In [ ]:
ws = conn.connect_websocket()
await ws.connect()
print("WebSocket connection_id:", ws.connection_id)

received_events = []
alert_events = []
scene_index_events = []

async def collect_websocket_events(ws, max_events=100):
    """Collect WebSocket messages in the background for inspection."""
    try:
        async for event in ws.receive():
            received_events.append(event)
            event_type = event.get("type") or event.get("event_type") or event.get("channel")

            if event_type == "alert" or "alert" in event:
                alert_events.append(event)
                print("\n🚨 Alert event received")
                print(json.dumps(event, indent=2)[:4000])
            elif event_type in {"scene_index", "rtstream_scene_index"}:
                scene_index_events.append(event)
                print("\n🧠 Scene index event received")
                print(json.dumps(event, indent=2)[:2000])
            else:
                print("\n📩 WebSocket event received")
                print(json.dumps(event, indent=2)[:2000])

            if len(received_events) >= max_events:
                break
    except asyncio.CancelledError:
        print("WebSocket collector stopped")
    except Exception as exc:
        print("WebSocket collector error:", repr(exc))

collector_task = asyncio.create_task(collect_websocket_events(ws))
print("Started background WebSocket collector")

## 📹 6. Connect a new RTStream sample

We use the known working intruder sample from the RTStream Cookbook. The stream name includes a timestamp so every run creates a new RTStream.

In [ ]:
RTSP_URL = "rtsp://samples.rts.videodb.io:8554/intruder"
run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
STREAM_NAME = f"Sandbox RTStream WebSocket Intruder Test {run_id}"

rtstream = coll.connect_rtstream(name=STREAM_NAME, url=RTSP_URL)
print("Connected new RTStream:", rtstream.id)
print(rtstream)

## 🧠 7. Create a sandbox-backed visual index

This is the important SDK call. It uses:

- `model_name=SELECTED_MODEL`
- `sandbox_id=sandbox.id`
- `ws_connection_id=ws.connection_id`


In [ ]:
INDEX_NAME = f"Sandbox_RTStream_WebSocket_Visual_Index_{run_id}"

scene_index = rtstream.index_visuals(
    batch_config={
        "type": "time",
        "value": 5,
        "frame_count": 2,
    },
    prompt=(
        "Monitor the area around the house. Describe whether a person is visible, "
        "where they are, and whether they appear to be loitering, approaching the door, "
        "checking the lock, or entering the property. If nothing suspicious is visible, "
        "say the property appears safe."
    ),
    model_name=SELECTED_MODEL,
    sandbox_id=sandbox.id,
    name=INDEX_NAME,
    ws_connection_id=ws.connection_id,
)

print("Created scene index:", scene_index.rtstream_index_id)
print("Index status:", getattr(scene_index, "status", None))
print("Index sandbox_id:", getattr(scene_index, "sandbox_id", None))

## 🚨 8. Create alert rules over WebSocket

The alerts are routed to the notebook with `ws_connection_id`. `callback_url=None` means no webhook service is required.

In [ ]:
EVENT_DEFINITIONS = [
    {
        "label": "person_loitering_near_property",
        "event_prompt": "Detect if a person is visible loitering near the house or property perimeter.",
    },
    {
        "label": "person_at_door_or_lock",
        "event_prompt": "Detect if a person approaches the door, interacts with the door, or appears to check the lock.",
    },
    {
        "label": "person_enters_property",
        "event_prompt": "Detect if a person crosses the property boundary or enters the house.",
    },
]

created_events = []
for spec in EVENT_DEFINITIONS:
    event_id = conn.create_event(
        event_prompt=spec["event_prompt"],
        label=f"{spec['label']}_{run_id}",
    )
    created_events.append({**spec, "event_id": event_id})
    print(f"Created event: {spec['label']} -> {event_id}")

In [ ]:
created_alerts = []

for event in created_events:
    alert_id = scene_index.create_alert(
        event_id=event["event_id"],
        callback_url=None,
        ws_connection_id=ws.connection_id,
    )
    created_alerts.append({**event, "alert_id": alert_id})
    print(f"Created WebSocket alert: {event['label']} -> {alert_id}")

## ✅ 9. Check configured alerts

In [ ]:
alerts = scene_index.list_alerts()
print(f"Configured alerts: {len(alerts)}")

for alert in alerts:
    print("-" * 80)
    print("Alert ID:", alert.get("alert_id"))
    print("Event ID:", alert.get("event_id"))
    print("Label:", alert.get("label"))
    print("Status:", alert.get("status"))
    print("Prompt:", alert.get("prompt"))

## 📩 10. Wait for WebSocket alerts

RTStream alerts are generated as the stream is indexed. Let this run for a few windows. If no alert arrives immediately, wait longer or re-run this cell.

In [ ]:
async def wait_for_alerts(timeout_seconds=180, min_alerts=1):
    start_count = len(alert_events)
    deadline = asyncio.get_event_loop().time() + timeout_seconds

    while asyncio.get_event_loop().time() < deadline:
        new_alerts = len(alert_events) - start_count
        if new_alerts >= min_alerts:
            print(f"Received {new_alerts} new alert event(s).")
            return alert_events[-new_alerts:]
        await asyncio.sleep(2)

    print(f"No new alert events after {timeout_seconds} seconds.")
    print(f"Total WebSocket events received: {len(received_events)}")
    print(f"Scene-index events received: {len(scene_index_events)}")
    print(f"Alert events received: {len(alert_events)}")
    return []

new_alerts = await wait_for_alerts(timeout_seconds=180, min_alerts=1)
new_alerts

## 🔎 11. Inspect the latest alert payload

In [ ]:
def compact_alert(event):
    """Print useful alert fields without assuming an exact payload shape."""
    payload = event.get("alert") or event.get("data") or event
    print(json.dumps(payload, indent=2)[:6000])

    stream_url = payload.get("stream_url") or payload.get("url")
    if stream_url:
        print("\nStream URL:", stream_url)

if alert_events:
    compact_alert(alert_events[-1])
else:
    print("No alert events captured yet. Re-run the wait cell above.")

## 🧹 12. Cleanup

Run this cell when you are done. It disables alerts, stops the scene index, stops the RTStream, closes the WebSocket, and stops the sandbox to avoid ongoing runtime charges.

In [ ]:
# Disable alerts
for alert in globals().get("created_alerts", []):
    try:
        scene_index.disable_alert(alert["alert_id"])
        print("Disabled alert:", alert["alert_id"])
    except Exception as exc:
        print("Could not disable alert", alert.get("alert_id"), repr(exc))

# Stop the scene index
try:
    scene_index.stop()
    print("Stopped scene index:", scene_index.rtstream_index_id)
except Exception as exc:
    print("Could not stop scene index:", repr(exc))

# Stop the RTStream
try:
    rtstream.stop()
    print("Stopped RTStream:", rtstream.id)
except Exception as exc:
    print("Could not stop RTStream:", repr(exc))

# Stop WebSocket collector and close the connection
try:
    collector_task.cancel()
    await asyncio.sleep(0)
except Exception as exc:
    print("Could not cancel collector task:", repr(exc))

try:
    await ws.close()
    print("Closed WebSocket")
except Exception as exc:
    print("Could not close WebSocket:", repr(exc))

# Stop the sandbox last. Use grace=True so in-flight work can finish cleanly.
try:
    sandbox.stop(grace=True)
    print(f"Stopping sandbox {sandbox.id}; current status: {sandbox.status}")
    sandbox.wait_for_stop(timeout=180, interval=5)
    print(f"Sandbox {sandbox.id} final status: {sandbox.status}")
except Exception as exc:
    print("Could not stop sandbox:", repr(exc))